<a href="https://colab.research.google.com/github/Penguin-IT/AI/blob/main/03.A*/BTVN/BaiTapVeNha_A*.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import heapq
from math import sqrt

class Diem:
    def __init__(self, x, y):
        self.toaDoX = x
        self.toaDoY = y

    def khoangCach(self, diemKhac):
        return sqrt((self.toaDoX - diemKhac.toaDoX)**2 + (self.toaDoY - diemKhac.toaDoY)**2)

    def __eq__(self, other):
        return self.toaDoX == other.toaDoX and self.toaDoY == other.toaDoY

    def __hash__(self):
        return hash((self.toaDoX, self.toaDoY))

    def __str__(self):
        return f"({self.toaDoX}, {self.toaDoY})"

class DuongDi:
    def __init__(self, diemHienTai, diemCha, tongChiPhi, hamUocLuong):
        self.diemHienTai = diemHienTai
        self.diemCha = diemCha
        self.tongChiPhi = tongChiPhi
        self.hamUocLuong = hamUocLuong

    def __lt__(self, other):
        return self.tongChiPhi + self.hamUocLuong < other.tongChiPhi + other.hamUocLuong

class BanDo:
    def __init__(self, diemBatDau, diemKetThuc, danhSachTru):
        self.diemBatDau = diemBatDau
        self.diemKetThuc = diemKetThuc
        self.danhSachTru = set(danhSachTru)

    def laTru(self, diem):
        return diem in self.danhSachTru

    def layHangXom(self, diem):
        hangXom = []
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0:
                    continue
                diemMoi = Diem(diem.toaDoX + dx, diem.toaDoY + dy)
                if not self.laTru(diemMoi):
                    hangXom.append(diemMoi)
        return hangXom

def timDuongNganNhatAStar(banDo):
    duongMo = []
    duongDaKhamPha = set()

    diemKhoiTao = DuongDi(banDo.diemBatDau, None, 0, banDo.diemBatDau.khoangCach(banDo.diemKetThuc))
    heapq.heappush(duongMo, diemKhoiTao)

    while duongMo:
        duongHienTai = heapq.heappop(duongMo)

        if duongHienTai.diemHienTai == banDo.diemKetThuc:
            duongDi = []
            while duongHienTai:
                duongDi.append(duongHienTai.diemHienTai)
                duongHienTai = duongHienTai.diemCha
            return duongDi[::-1]

        duongDaKhamPha.add(duongHienTai.diemHienTai)

        for diemHangXom in banDo.layHangXom(duongHienTai.diemHienTai):
            if diemHangXom in duongDaKhamPha:
                continue

            chiPhiMoi = duongHienTai.tongChiPhi + 1
            hamUocLuongMoi = diemHangXom.khoangCach(banDo.diemKetThuc)

            duongMoi = DuongDi(diemHangXom, duongHienTai, chiPhiMoi, hamUocLuongMoi)

            for duongTrongHangDoi in duongMo:
                if duongTrongHangDoi.diemHienTai == diemHangXom and \
                   duongTrongHangDoi.tongChiPhi + duongTrongHangDoi.hamUocLuong <= \
                   duongMoi.tongChiPhi + duongMoi.hamUocLuong:
                    break
            else:
                heapq.heappush(duongMo, duongMoi)

    return None

def hienThiBanDo(banDo, duongTimDuoc):
    minX = min(d.toaDoX for d in banDo.danhSachTru | {banDo.diemBatDau, banDo.diemKetThuc})
    maxX = max(d.toaDoX for d in banDo.danhSachTru | {banDo.diemBatDau, banDo.diemKetThuc})
    minY = min(d.toaDoY for d in banDo.danhSachTru | {banDo.diemBatDau, banDo.diemKetThuc})
    maxY = max(d.toaDoY for d in banDo.danhSachTru | {banDo.diemBatDau, banDo.diemKetThuc})

    for y in range(int(maxY), int(minY) - 1, -1):
        dong = ""
        for x in range(int(minX), int(maxX) + 1):
            diem = Diem(x, y)
            if duongTimDuoc and diem in duongTimDuoc:
                dong += "*"
            elif diem == banDo.diemBatDau:
                dong += "S"
            elif diem == banDo.diemKetThuc:
                dong += "E"
            elif banDo.laTru(diem):
                dong += "#"
            else:
                dong += " "
        print(dong)
    print(f"Độ dài đường đi: {len(duongTimDuoc)-1 if duongTimDuoc else 'Không tìm thấy'}")

def main():
    diemBatDau = Diem(1, 1)
    diemKetThuc = Diem(18, 10)

    danhSachTru = [
        Diem(5, 2), Diem(5, 3), Diem(5, 4), Diem(5, 5), Diem(5, 6),
        Diem(9, 4), Diem(9, 5), Diem(9, 6), Diem(9, 7),
        Diem(13, 2), Diem(13, 3), Diem(13, 8), Diem(13, 9),
        Diem(3, 7), Diem(4, 7), Diem(7, 9), Diem(8, 9),
        Diem(11, 1), Diem(15, 5), Diem(16, 5), Diem(17, 5)
    ]

    banDo = BanDo(diemBatDau, diemKetThuc, danhSachTru)

    print("Bản đồ ban đầu:")
    hienThiBanDo(banDo, [])

    duongTimDuoc = timDuongNganNhatAStar(banDo)

    print("\nĐường đi tìm được:")
    hienThiBanDo(banDo, duongTimDuoc)

if __name__ == "__main__":
    main()

Bản đồ ban đầu:
                 E
      ##    #     
            #     
  ##    #         
    #   #         
    #   #     ### 
    #   #         
    #       #     
    #       #     
S         #       
Độ dài đường đi: Không tìm thấy

Đường đi tìm được:
         *********
      ##*   #     
     ***    #     
  ##*   #         
   *#   #         
   *#   #     ### 
   *#   #         
  * #       #     
 *  #       #     
*         #       
Độ dài đường đi: 19
